# 6. DSMS Apps and Pipelines

In this tutorial we see how to create apps and run them manually

### 6.1. Setting up

Before you run this tutorial: make sure to have access to a DSMS-instance of your interest, along with installation of this package, and have established access to the DSMS through DSMS-SDK (refer to [Connecting to DSMS](../dsms_sdk.md#connecting-to-dsms))


Now let us import the needed classes and functions for this tutorial.

In [1]:
from dsms import DSMS, KItem, AppConfig
import time

Now source the environmental variables from an `.env` file and start the DSMS-session.

In [2]:
import os
dsms = DSMS(env=".env") if os.path.exists(".env") else DSMS()

### 6.2. Investigating Available Apps

We can investigate which apps are available:

In [3]:
dsms.app_configs

[app:
   name: SD_Tensile_Test_Pipeline,
 app:
   name: csv_tensile_test,
 app:
   name: csv_tensile_test_f2,
 app:
   name: dsms-materialcard,
 app:
   name: dsms-tensile-test-analysis,
 app:
   name: excel_notch_tensile_test,
 app:
   name: excel_shear_tensile_test,
 app:
   name: excel_tensile_test]

### 6.3. Create a new app config and apply it to a KItem

### 6.3.1. Data2RDF

#### 6.3.1.1 Prepare app and its config

In the following example, we would like to upload some csv with some arbitrary data and describe it through an RDF. This will give us the opportunity to harmonize the entities of the data file through ontological concepts and allow us to convert values of the data frame columns in to any compatible unit we would like to have.

Fist of all, let us define the data:

In [4]:
data = """A,B,C
1.2,1.3,1.5
1.7,1.8,1.9
2.0,2.1,2.3
2.5,2.6,2.8
3.0,3.2,3.4
3.6,3.7,3.9
4.1,4.3,4.4
4.7,4.8,5.0
5.2,5.3,5.5
5.8,6.0,6.1
"""

We will also give the config a defined name:

In [5]:
configname = "testapp2"

As a next step, we want to create a new app specification. The specification is following the definition of an [**Argo Workflow**](https://argo-workflows.readthedocs.io/en/latest/). The workflow shall trigger a pipeline from a docker image with the  [**Data2RDF package**](https://data2rdf.readthedocs.io/en/latest/). 

The image has already been deployed on the k8s cluster of the DSMS and the workflow template with the name `dsms-data2rdf` has been implemented previously. Hence we only need to configure our pipeline for our data shown above, which we would like to upload and describe through an RDF.

For more details about the data2rdf package, please refer to the documentation of Data2RDF mentioned above.

The parameters of the app config are defining the inputs for our Data2RDF pipeline. This e.g. are: 

* the parser kind (`csv` here)
* the time series header length (`1` here)
* the metadata length (`0` here)
* the time series separator (`,` here)
* the log level (`DEBUG` here)
* the mapping 
    * `A` is the test time and has a unit in seconds
    * `B` is the standard force in kilonewtons
    * `C` is the absolut cross head travel in millimeters

In [6]:
parameters = [
    {"name": "parser", "value": "csv"},
    {"name": "time_series_header_length", "value": 1},
    {"name": "metadata_length", "value": 0},
    {"name": "time_series_sep", "value": ","},
    {
        "name": "mapping",
        "value": """
            [
                {
                    "key": "A",
                    "iri": "https://w3id.org/steel/ProcessOntology/TestTime",
                    "unit": "s"
                },
                {
                    "key": "B",
                    "iri": "https://w3id.org/steel/ProcessOntology/StandardForce",
                    "unit": "kN"
                },
                {
                    "key": "C",
                    "iri": "https://w3id.org/steel/ProcessOntology/AbsoluteCrossheadTravel",
                    "unit": "mm"
                }
            ]
            """,
    },
]

Now we add the parameters to our app specification. We assign a prefix `datardf-` which shall generate a new name with some random characters as suffix. The workflow template with the Docker image we want to run is called `dsms-data2rdf` and its `entrypoint` is `execute_pipeline`.

In [7]:
# Define app specification
specification = {
    "apiVersion": "argoproj.io/v1alpha1",
    "kind": "Workflow",
    "metadata": {"generateName": "data2rdf-"},
    "spec": {
        "entrypoint": "execute-pipeline",
        "workflowTemplateRef": {"name": "dsms-data2rdf-1.2.5"},
        "arguments": {"parameters": parameters},
    },
}

Now we instantiate the new app config:

In [8]:
appspec = AppConfig(
    name=configname,
    specification=specification,  # this can also be a file path to a yaml file instead of a dict
    expose_sdk_config=True,
)


Please note that the `expose_sdk_config` is important here, since the pipeline needs to be aware of the settings of our platform and our current SDK-session.

We commit the new app config:

In [9]:
dsms.add(appspec)
dsms.commit()

/root/dsms/dsms-python-sdk/dsms/apps/config.py:86: UserWarning: AppConfigs do not have a refresh functionality since they are already up to date after committing. You can continue normally using the app config.
  warnings.warn(


And show the app specification:

In [10]:
appspec.specification

{'apiVersion': 'argoproj.io/v1alpha1',
 'kind': 'Workflow',
 'metadata': {'generateName': 'data2rdf-'},
 'spec': {'entrypoint': 'execute-pipeline',
  'workflowTemplateRef': {'name': 'dsms-data2rdf-1.2.5'},
  'arguments': {'parameters': [{'name': 'parser', 'value': 'csv'},
    {'name': 'time_series_header_length', 'value': 1},
    {'name': 'metadata_length', 'value': 0},
    {'name': 'time_series_sep', 'value': ','},
    {'name': 'mapping',
     'value': '\n            [\n                {\n                    "key": "A",\n                    "iri": "https://w3id.org/steel/ProcessOntology/TestTime",\n                    "unit": "s"\n                },\n                {\n                    "key": "B",\n                    "iri": "https://w3id.org/steel/ProcessOntology/StandardForce",\n                    "unit": "kN"\n                },\n                {\n                    "key": "C",\n                    "iri": "https://w3id.org/steel/ProcessOntology/AbsoluteCrossheadTravel",\n    

Now we would like to apply the app config to a KItem. `triggerUponUpload` must be set to `True` so that the app is triggered automatically when we upload an attachment.

Additionally, we must tell the file extension for which the upload shall be triggered. Here it is `.csv`.

We also want to generate a qr code as avatar for the KItem with `avatar={"include_qr": True}`.

In [11]:
item = KItem(
    name="my tensile test experiment",
    ktype_id=dsms.ktypes.Dataset,
    apps=[
        {
            "executable": appspec.name,
            "title": "data2rdf",
            "additional_properties": {
                "triggerUponUpload": True,
                "triggerUponUploadFileExtensions": [".csv"],
            },
        }
    ],
    avatar={"include_qr": True},
)

We commit the KItem:

In [12]:
dsms.add(item)
dsms.commit()

Now we add our data with our attachment:

In [13]:
item.attachments = [{"name": "dummy_data.csv", "content": data}]

And we commit again:

In [14]:
dsms.add(item)
dsms.commit()

#### 6.3.1.2 Get results

> **Note:** Some fields visible in KItem outputs (`authors`, `rdf_exists`, `user_groups`) are deprecated in v5.0.0 and are no longer populated by the server. They remain in the model for backward compatibility. Use `access_properties` for access control.

Now we can verify that the data extraction was successful:

In [15]:
item.refresh()

In [16]:
print(item)

kitem:
  id: 2ef6d03a-099e-40c4-a508-c90b5c85789f
  name: my tensile test experiment
  ktype_id: dataset
  slug: mytensiletestexperiment-2ef6d03a
  avatar_exists: false
  has_contexts: false
  annotations: []
  attachments:
  - name: dummy_data.csv
  linked_kitems: []
  affiliations: []
  authors: []
  contacts: []
  created_at: 2026-06-07 20:53:04.030500
  updated_at: 2026-06-07 20:53:04.030500
  external_links: []
  apps:
  - executable: testapp2
    title: data2rdf
    description: null
    tags: null
    additional_properties:
      triggerUponUpload: true
      triggerUponUploadFileExtensions:
      - .csv
  rdf_exists: false
  access_properties:
    visibility: private
    user_access:
    - role: OWNER
      user_id: 6be66d9f-1a9f-44fc-8176-f71155de06ba
    group_access: []
  contexts: []



And also that the RDF generation was successful:

In [17]:
try:
    print(item.subgraph.serialize())
except ValueError as e:
    print(f"Note: RDF subgraph is generated asynchronously.")
    print(f"It may not be available immediately after creation.")

Note: RDF subgraph is generated asynchronously.
It may not be available immediately after creation.


And now we are able to convert our data into any compatiable unit we want. For the `StandardForce`, it was previously `kN`, but we want to have it in `N` now:


In [18]:
try:
    item.dataframe.StandardForce.convert_to("N")
except (AttributeError, ValueError) as e:
    print(f"Note: dataframe may not be available yet (app processing is asynchronous).")
    print(f"Error: {e}")

Note: dataframe may not be available yet (app processing is asynchronous).
Error: 'NoneType' object has no attribute 'StandardForce'


#### 6.3.1.3 Manipulate dataframe

We are able to retrieve the dataframe as pd.DataFrame:

In [19]:
try:
    item.dataframe.to_df()
except (AttributeError, ValueError) as e:
    print(f"Note: dataframe not available (asynchronous processing).")
    print(f"Error: {e}")

Note: dataframe not available (asynchronous processing).
Error: 'NoneType' object has no attribute 'to_df'


We are able to overwrite the dataframe with new data:

In [20]:
item.dataframe = {
    "TestTime": list(range(100)),
    "StandardForce": list(range(1,101)),
    "AbsoluteCrossheadTravel": list(range(2,102))
}
dsms.add(item)
dsms.commit()

We are able to retrieve the data colum-wise:

In [21]:
try:
    for column in item.dataframe:
        print("column:", column.name, ",\n", "data:", column.get())
except (AttributeError, TypeError) as e:
    print(f"Note: dataframe not available yet.")
    print(f"Error: {e}")

column: TestTime ,
 data: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


column: StandardForce ,
 data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]
column: AbsoluteCrossheadTravel ,
 data: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101]


... and also to modify the dataframe directly as we need:

In [22]:
try:
    new_df = item.dataframe.to_df().drop(['TestTime'], axis=1)
    item.dataframe = new_df
except (AttributeError, TypeError) as e:
    print(f"Note: dataframe not available yet.")
    print(f"Error: {e}")

#### 6.3.1.4 Run app on demand

We are able to run the app on demand, not being triggered automatically during the upload of an attachment every time. For this purpose, we just need to refer to the name of the app we assigned during the KItem creation ( here it is simply `data2rdf`).

Additionally, we need to tell the `attachment_name` and hand over the access token and host url to the app by explicitly setting `expose_sdk_config` to `True`. This is basically telling the SDK that the app is using the SDK also internally and that the app should receive its parameters from the current SDK session.

The app is running synchronously, hence the `job` is created when the pipeline run finished.

In [23]:
try:
    job = item.apps.by_title["data2rdf"].run(
        attachment_name="dummy_data.csv",
        expose_sdk_config=True
    )
except RuntimeError as e:
    print(f"Note: app execution requires appropriate permissions.")
    print(f"Error: {e}")
    job = None

Note: app execution requires appropriate permissions.
Error: Submission was not successful: {"detail":{"message":"Workflow could not be executed: Server returned status code 401 with message: `Unauthorized`."}}


We are able to retrieve the job status:

In [24]:
print(job.status) if job else print("Job not available.")

Job not available.


... and the job logs:

In [25]:
print(job.logs) if job else print("Job not available.")

Job not available.


In case we would like to run the job in the background, we simply add a `wait=False`:

In [26]:
try:
    job = item.apps.by_title["data2rdf"].run(
        attachment_name="dummy_data.csv",
        expose_sdk_config=True,
        wait=False,
    )
except RuntimeError as e:
    print(f"Note: app execution requires appropriate permissions.")
    print(f"Error: {e}")
    job = None

Note: app execution requires appropriate permissions.
Error: Submission was not successful: {"detail":{"message":"Workflow could not be executed: Server returned status code 401 with message: `Unauthorized`."}}


We are able to monitor the job status and logs asynchronously:

In [27]:
if job:
    import time
    while True:
        time.sleep(1)
        print(job.status)
        print("Current logs:")
        print(job.logs)
        print("\n")
        if job.status.phase != "Running":
            break
else:
    print("Job not available.")

Job not available.


**IMPORTANT**: When job has run asychronously (in the background), we need to manually refresh the KItem afterwards:

In [28]:
item.refresh()

Clean up the DSMS from the tutorial

In [29]:
del dsms[item]
del dsms[appspec]
dsms.commit()